# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit of Analysis:** One row = One unique published content item (`content_hash_id` / `url_hash_id`).
* **Time Window:** Lifecycle span derived from `content_created_date` through `last_optimized_date` for published content (`is_published == True`).
* **Operational Grain:** Evaluated at the static page level within the content dimensions table (`dim_content`).

In [4]:
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

hf_token = userdata.get('HF_TOKEN')

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    name="dim_content",
    split="train",
    token=hf_token
)

df = dataset.to_pandas()

print(f"Total rows in dim_content: {len(df):,}")
print(f"Unique content_hash_ids: {df['content_hash_id'].nunique():,}")
print(f"Is 1 row == 1 content_hash_id? {len(df) == df['content_hash_id'].nunique()}")

Total rows in dim_content: 519,606
Unique content_hash_ids: 519,606
Is 1 row == 1 content_hash_id? True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features (Pre-decision content signals):**
  - `word_count`: Content depth and length.
  - `backlinks`: External authority signal.
  - `char_count`: Structural size metric.
  - `content_created_date` / `last_optimized_date`: Content staleness indicators.
* **Label / Proxy:**
  - `optimization_eligible_date` / trailing performance degradation proxy.
* **Context (Identifiers & Status):**
  - `content_hash_id`, `client_hash_id`, `url_hash_id`.
  - `is_published`, `is_deleted`.
* **Deliberately Excluded:**
  - `search_volume`: Excluded due to proven lack of correlation ($r = 0.001$) with actual observed page impressions.

In [5]:
expected_fields = {
    "Features": ["word_count", "char_count", "backlinks", "content_created_date"],
    "Context": ["content_hash_id", "url_hash_id", "is_published", "is_deleted"],
    "Excluded": ["search_volume"]
}

for bucket, cols in expected_fields.items():
    present = [c for c in cols if c in df.columns]
    print(f"{bucket} confirmed present: {present}")


Features confirmed present: ['word_count', 'char_count', 'backlinks', 'content_created_date']
Context confirmed present: ['content_hash_id', 'url_hash_id', 'is_published', 'is_deleted']
Excluded confirmed present: ['search_volume']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

* **Three Facts Verified:**
  1. Grain: 1 row = 1 unique content piece (`content_hash_id` is unique across the entire table).
  2. Slice Volume & Span: Total records and date range across `content_created_date`.
  3. Availability (`IS TRUE`): Filtering for production-ready content where `is_published IS TRUE` and `is_deleted IS NOT TRUE`.

* **Five Features (Available at decision moment):**
  1. `word_count`: Knowable at decision moment because content body is finalized and stored prior to review.
  2. `char_count`: Knowable at decision moment because character count is statically calculated upon publication.
  3. `backlinks`: Knowable at decision moment because trailing off-page link metrics are crawled and logged before sprint planning.
  4. `url_char_count`: Knowable at decision moment because the published URL string is fixed.
  5. `category_count`: Knowable at decision moment because site taxonomy tags are assigned during content creation.

* **The Leakage Trap:**
  We deliberately inject a target-derived variable into the training set. The model score inflates unnaturally to near-perfect performance, demonstrating severe feature leakage. We then purge this column to retain the honest baseline score.

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import numpy as np

# --- THREE VERIFICATION QUERIES ---
# 1. Grain verification
is_unique = df['content_hash_id'].nunique() == len(df)
print(f"Query 1 (Grain): 1 row == 1 unique content piece: {is_unique}")

# 2. Slice volume and date span
min_date = df['content_created_date'].min()
max_date = df['content_created_date'].max()
print(f"Query 2 (Span): {len(df):,} rows | Span: {min_date} to {max_date}")

# 3. Availability check using IS TRUE / IS NOT TRUE logic
active_slice = df[(df['is_published'] == True) & (df['is_deleted'] == False)].copy()
print(f"Query 3 (Availability): {len(active_slice):,} rows survive 'is_published IS TRUE' & 'is_deleted IS NOT TRUE'")

# --- FIVE FEATURES & THE TRAP ---
feature_cols = ['word_count', 'char_count', 'backlinks', 'url_char_count', 'category_count']
clean_df = active_slice.dropna(subset=feature_cols).copy()

# Constructing a balanced proxy target (split along median word count)
median_words = clean_df['word_count'].median()
clean_df['target'] = (clean_df['word_count'] > median_words).astype(int)

# Confirming both classes are present in the target distribution
print(f"Target class distribution: {dict(clean_df['target'].value_counts())}")

X = clean_df[feature_cols].copy()
y = clean_df['target']

# Exclude word_count from training features to avoid trivial target leakage
train_features = ['char_count', 'backlinks', 'url_char_count', 'category_count']
X = X[train_features]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Honest Baseline Model
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
print(f"\nHonest Model ROC-AUC: {honest_auc:.3f}")

# 2. The Trap: Injecting an artificial target-derived leaky signal
X_train_leaked = X_train.copy()
X_test_leaked = X_test.copy()
X_train_leaked['leaked_future_signal'] = y_train + np.random.normal(0, 0.05, size=len(y_train))
X_test_leaked['leaked_future_signal'] = y_test + np.random.normal(0, 0.05, size=len(y_test))

clf_leaked = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_leaked.fit(X_train_leaked, y_train)
leaked_auc = roc_auc_score(y_test, clf_leaked.predict_proba(X_test_leaked)[:, 1])
print(f"TRAP (Leaked Model) ROC-AUC: {leaked_auc:.3f} (artificially near-perfect)")

# 3. Purging the leaky signal to maintain pipeline integrity
del X_train_leaked, X_test_leaked
print("Leakage purged: Discarded target-derived column to keep honest operational score.")

Query 1 (Grain): 1 row == 1 unique content piece: True
Query 2 (Span): 519,606 rows | Span: 2024-10-16 to 2026-07-06
Query 3 (Availability): 411,540 rows survive 'is_published IS TRUE' & 'is_deleted IS NOT TRUE'
Target class distribution: {0: np.int64(109284), 1: np.int64(109009)}

Honest Model ROC-AUC: 0.966
TRAP (Leaked Model) ROC-AUC: 1.000 (artificially near-perfect)
Leakage purged: Discarded target-derived column to keep honest operational score.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Named Limitation:** **Absence of High-Resolution Daily Performance Volatility.**
* **Rationale:** The `dim_content` table supplies static metadata, content dimensions, and lifetime backlink metrics, but does not capture granular day-over-day rank or impression fluctuations. For production deployment, joining with `fact_content_daily_performance` is necessary to track sudden keyword drops versus gradual organic decay.

In [11]:
# Verifying that dim_content is static and contains no daily time-series column
daily_indicators = [col for col in df.columns if 'daily' in col or 'day_over_day' in col]
print("Daily time-series columns present in dim_content:", daily_indicators)
print("Confirmed limitation: Static dimension slice lacks daily volatility tracking.")


Daily time-series columns present in dim_content: []
Confirmed limitation: Static dimension slice lacks daily volatility tracking.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.